# Caras lab ephys analysis pipeline
This pipeline is intended to be run after extracting behavioral timestamps and neuron spike times with our [MatLab pipeline](https://github.com/caraslab/caraslab-spikesortingKS2)

Files need to be organized in a specific folder structure or file paths need to be changed

File structure can be found in the Sample dataset folder

## Imports
Specific imports can be found within each function

In [1]:
%load_ext IPython.extensions.autoreload
%autoreload 2

import warnings
from os.path import sep
from glob import glob

from matplotlib.pyplot import rcParams
import helpers.get_JSON_data as get_JSON_data
import helpers.confirm_run as confirm_run
import outcome_decoding_analysis.outcome_decoding_atemporal as outcome_decoding_atemporal
import outcome_decoding_analysis.outcome_decoding_future_trial as outcome_decoding_future_trial
import outcome_decoding_analysis.outcome_decoding_timeSeries as outcome_decoding_timeSeries

warnings.filterwarnings("ignore")
# Some plotting parameters
label_font_size = 11
tick_label_size = 7
legend_font_size = 6
line_thickness = 1

rcParams['figure.dpi'] = 600
rcParams['pdf.fonttype'] = 42
rcParams['ps.fonttype'] = 42
rcParams['font.family'] = 'Arial'
rcParams['font.weight'] = 'regular'
rcParams['axes.labelweight'] = 'regular'

rcParams['font.size'] = label_font_size
rcParams['axes.labelsize'] = label_font_size
rcParams['axes.titlesize'] = label_font_size
rcParams['axes.linewidth'] = line_thickness
rcParams['legend.fontsize'] = legend_font_size
rcParams['xtick.labelsize'] = tick_label_size
rcParams['ytick.labelsize'] = tick_label_size
rcParams['errorbar.capsize'] = label_font_size
rcParams['lines.markersize'] = line_thickness
rcParams['lines.linewidth'] = line_thickness

## Set global paths and variables

FILE NAMING REQUIREMENTS

This pipeline matches files using filenames. The date and time embedded in each
behavioral filename are detected automatically by content (a 6-digit YYMMDD-HHMMSS
run, or a dashed YYYY-MM-DD/HH-MM-SS pair), regardless of how many other fields
precede them in the filename — so you don't need to configure their position.

Here are the defaults that come out of the MatLab processing pipeline:
- Synapse:
    - Behavior file: SUBJ-ID-154_MML-Aversive-AM-210501-112033_trialInfo.csv
    - Spike time file: SUBJ-ID-154_210501_concat_cluster2627.txt
- Intan:
    - Behavior file: SUBJ-ID-231_2021-07-17_15-19-28_Active_trialInfo.csv
    - Spike time file: SUBJ-ID-231_210717_concat_cluster651.txt

If you need to alter the filename matching structure, edit this function: helpers.preprocess_files.extract_session_key

In [18]:
DATA_PATH = r'.' + sep + 'Sample_data'

SETTINGS_DICT = {
    'EXPERIMENT_TAG': 'OFCPL',  # Appends to start of summary files
    'SPIKES_PATH': DATA_PATH + sep + 'Spike times',
    'KEYS_PATH': DATA_PATH + sep + 'Key files',
    'OUTPUT_PATH': DATA_PATH + sep + 'Output',

    'SESSIONS_TO_RUN': None,

    'SESSIONS_TO_EXCLUDE': None,

    #########################################
    # Parameters for Nth trial decoder
    'DECODER_N_FILE_NAME_TAG':       'Outcome_allUnits_responseSpikes_lda',

    'DECODER_N_METHOD':              'lda',
    'DECODER_N_SESSION':             'active',
    'DECODER_N_SPIKES_FIELDNAME':    'Response_spikes',  # Irrelevant if reading from csv
    'DECODER_N_SPIKE_TIME_FORMAT':   True,  # Irrelevant if reading from csv
    'DECODER_N_START_TIME':           -2.0,
    'DECODER_N_END_TIME':             3.0,
    'DECODER_N_BIN_SIZE':             0.1,
    'DECODER_N_PSTH_STEP_SIZE':       0.1,
    'DECODER_N_EPOCH_START':          0.3,
    'DECODER_N_EPOCH_END':            3,
    'DECODER_N_GAUSSIAN_SIGMA':       0,
    'DECODER_N_SHOCK_ARTIFACT':       None,
    'DECODER_N_N_CV_SPLITS':          3,
    'DECODER_N_RANDOM_STATE':         0,
    'DECODER_N_SUBJECTS':            None,
    'DECODER_N_SESSIONS_TO_EXCLUDE': None,
    'DECODER_N_UNITS_CSV':           None,  # DATA_PATH + sep + sep.join(['Output', 'SU_list.csv']),
    'DECODER_N_THRESHOLD_CSV':       None,
    'DECODER_N_OUTPUT_FOLDER':       None,

    'DECODER_N_DECODE_TARGET':       'outcome',   # str or list: 'outcome'|'amdepth'|'amdepth_hits'|'amdepth_misses'
    'DECODER_N_SHOCK_FLAG_FILTER':   1,           # scalar or list zipped with DECODER_N_DECODE_TARGET: 1 | 0 | None

    'DECODER_N_ISI_GRID_MIN':        0.001,           # seconds (1 ms)
    'DECODER_N_ISI_GRID_MAX':        2.0,             # seconds
    'DECODER_N_ISI_N_GRID':          20,              # log-spaced evaluation points
    'DECODER_N_MIN_TRIALS_PER_CLASS': 3,              # drop classes with fewer trials; 0 = disabled
                                                      # set equal to DECODER_N_N_CV_SPLITS to avoid
                                                      # session skips due to sparse depth classes

    'DECODER_N_FEATURE_MODE':        'full_raster',  # 'full_raster' | 'mean_rate' | 'isi' | 'csv'
    'DECODER_N_LOG1P_TRANSFORM':     False,           # apply log1p() to features before LDA;
                                                      # ignored for 'isi' and 'csv' feature modes

    # If using a CSV file, set these and change DECODER_N_FEATURE_MODE to csv
    'DECODER_N_CSV_FILE':            None,
    'DECODER_N_CSV_SAMPLING_RATE':   10,             # Hz — TP column sampling rate; set to 1/BIN_SIZE
    'DECODER_N_CSV_ANALYSIS_ID': 'Response_timeSeries_zscore_globalBaseline',

    #########################################
    # Parameters for Nth+1 trial decoder
    'DECODER_N1_METHOD':              'lda',
    'DECODER_N1_SESSION':             'active',
    'DECODER_N1_SPIKES_FIELDNAME':    'Response_spikes',
    'DECODER_N1_SPIKE_TIME_FORMAT':   True,
    'DECODER_N1_START_TIME':          -2.0,
    'DECODER_N1_END_TIME':             3.0,
    'DECODER_N1_BIN_SIZE':             0.1,
    'DECODER_N1_PSTH_STEP_SIZE':       0.1,
    'DECODER_N1_EPOCH_START':         -2.0,
    'DECODER_N1_EPOCH_END':            3.0,
    'DECODER_N1_GAUSSIAN_SIGMA':       0,
    'DECODER_N1_SHOCK_ARTIFACT':       None,
    'DECODER_N1_N_CV_SPLITS':              5,
    'DECODER_N1_RANDOM_STATE':             0,
    'DECODER_N1_MIN_PAIRS_STRATIFIED':    10,   # min valid pairs per trial-N-type subset
    'DECODER_N1_N_PERMUTATIONS':        1000,   # permutation test iterations for threshold transition
    'DECODER_N1_TESTS_TO_RUN':          [      # comment out any to skip
        'atemporal',         # trial N → trial N outcome (sanity check)
        'two_stage',         # trial N neural → stage1 signal → trial N+1 outcome
        'direct',            # trial N neural → trial N+1 outcome (single stage)
        'threshold',         # ShockFlag 1→0 transitions only + permutation test
        'by_ntype',          # direct decoder split by trial N type (Hit-N / Miss-N)
        'decision_variance',     # CV decision value variance per trial type (stereotypy probe)
        'decision_correlation',  # correlation between trial-N decision value and trial-N+1 outcome
        'behavioral_baseline',   # win-stay behavioral autocorrelation (no neural data)
    ],
    'DECODER_N1_SUBJECTS':            None,
    'DECODER_N1_SESSIONS_TO_EXCLUDE': None,
    'DECODER_N1_UNITS_CSV':           None,
    'DECODER_N1_THRESHOLD_CSV':       None,
    'DECODER_N1_OUTPUT_FOLDER':       None,
    'DECODER_N1_FILE_NAME_TAG':       'TrialType_HitVsMiss_lda_fullRaster_futureTrial',

    #########################################
    # Parameters for time series decoder
    'DECODER_TS_FILE_NAME_TAG':     'AMdepth_Hit_AMresponsiveSUs_TS_lda',
    'DECODER_TS_METHOD':            'lda',
    'DECODER_TS_SESSION':           'active',
    'DECODER_TS_SPIKES_FIELDNAME':  'Response_spikes',
    'DECODER_TS_SPIKE_TIME_FORMAT': True,
    'DECODER_TS_START_TIME':        -2.0,
    'DECODER_TS_END_TIME':           3.0,
    'DECODER_TS_BIN_SIZE':           0.1,
    'DECODER_TS_PSTH_STEP_SIZE':     0.1,
    'DECODER_TS_GAUSSIAN_SIGMA':     0,
    'DECODER_TS_SHOCK_ARTIFACT':    None,
    'DECODER_TS_DECODE_TARGET':      'amdepth_hits',   # 'outcome' | 'amdepth' | 'amdepth_hits' | 'amdepth_misses'
    'DECODER_TS_SHOCK_FLAG_FILTER':  None,           # 1 | 0 | None (all trials in selected subset)
    'DECODER_TS_ISI_GRID_MIN':       0.001,           # seconds (1 ms)
    'DECODER_TS_ISI_GRID_MAX':       2.0,             # seconds
    'DECODER_TS_ISI_N_GRID':         20,              # log-spaced evaluation points
    'DECODER_TS_MIN_TRIALS_PER_CLASS': 3,              # drop classes with fewer trials; 0 = disabled
                                                       # set equal to DECODER_TS_N_CV_SPLITS to avoid
                                                       # session skips due to sparse depth classes
    'DECODER_TS_N_HISTORY_BINS':     3,
    'DECODER_TS_N_CV_SPLITS':        3,
    'DECODER_TS_RANDOM_STATE':       0,
    'DECODER_TS_SUBJECTS':          None,
    'DECODER_TS_SESSIONS_TO_EXCLUDE': None,
    'DECODER_TS_UNITS_CSV':         None,
    'DECODER_TS_THRESHOLD_CSV':     None,
    'DECODER_TS_OUTPUT_FOLDER':     None,

    'DECODER_TS_FEATURE_MODE':        'mean_rate',  # 'full_raster' | 'mean_rate' | 'isi' | 'csv'
    'DECODER_TS_LOG1P_TRANSFORM':     True,         # apply log1p() to features before LDA;
                                                     # ignored for 'isi' and 'csv' feature modes

    # If using a CSV file, set these and change DECODER_TS_FEATURE_MODE to csv
    'DECODER_TS_CSV_FILE':            None,
    'DECODER_TS_CSV_SAMPLING_RATE':   10,             # Hz — TP column sampling rate; set to 1/BIN_SIZE
    'DECODER_TS_CSV_ANALYSIS_ID':     'Response_timeSeries_zscore_globalBaseline',
}

## Load JSON file names

In [19]:
# Load existing JSONs
json_filenames = glob(SETTINGS_DICT['OUTPUT_PATH'] + sep + 'JSON files' + sep + '*json')

# Retrieve data in JSON
print("Loading data in JSONs...")

filtered_files =  get_JSON_data.get_JSON_data(
    json_filenames,
    sessions_to_run=SETTINGS_DICT['SESSIONS_TO_RUN'],
    sessions_to_exclude=SETTINGS_DICT['SESSIONS_TO_EXCLUDE']
)

Loading data in JSONs...


# LDA discrimination

In [ ]:
def _run():
    """Run atemporal (single time-window) LDA/SVM decoding of trial outcome, per DECODER_N_* settings.

    Decodes a fixed epoch (DECODER_N_EPOCH_START..END) per session using the
    method/feature settings configured above, writing a CSV + PDF summary of
    decoding accuracy vs. chance under DECODER_N_OUTPUT_FOLDER.
    """
    outcome_decoding_atemporal.run(filtered_files, SETTINGS_DICT)

confirm_run.confirm_run("This will run atemporal LDA classification. Continue?", _run)

# LDA discrimination of trial N+1

In [ ]:
def _run():
    """Run the trial-N -> trial-N+1 decoding analyses, per DECODER_N1_* settings.

    Runs whichever tests are listed in DECODER_N1_TESTS_TO_RUN (atemporal
    sanity check, two-stage, direct, threshold-transition + permutation
    test, by-trial-N-type, decision-variance/correlation probes, and/or the
    behavioral win-stay baseline), writing a combined CSV + PDF summary
    under DECODER_N1_OUTPUT_FOLDER.
    """
    outcome_decoding_future_trial.run(filtered_files, SETTINGS_DICT)

confirm_run.confirm_run("This will run atemporal LDA classification. Continue?", _run)

# LDA discrimination of time series

In [ ]:
def _run():
    """Run per-time-bin LDA/SVM decoding of trial outcome across the session, per DECODER_TS_* settings.

    Decodes every bin from DECODER_TS_START_TIME to DECODER_TS_END_TIME
    (optionally concatenating DECODER_TS_N_HISTORY_BINS preceding bins as
    extra features), writing a CSV + PDF of decoding accuracy vs. chance
    over time under DECODER_TS_OUTPUT_FOLDER.
    """
    outcome_decoding_timeSeries.run(filtered_files, SETTINGS_DICT)

confirm_run.confirm_run("This will run atemporal LDA classification on time series. Continue?", _run)